# SigFlow-Sim — All-in-One Notebook

A leakage-safe volatility-forecasting pipeline with:

- independent chronological windows for every ticker;
- past-only feature normalisation;
- purged train/validation/test splits;
- log annualised realised-volatility targets;
- signature and statistical path features;
- supervised low/medium/high volatility regimes;
- a gate-weighted mixture of nonlinear spline-flow experts;
- early stopping, gradient clipping and checkpointing;
- probabilistic forecasts, calibration and CRPS evaluation;
- rolling-volatility and EWMA baselines;
- optional feature ablations and genuine walk-forward refitting.

Everything required to configure, train, evaluate, plot and save the experiment is contained in this notebook.

## Recommended run order

1. Run the environment-setup cell.
2. Review the `Config` cell.
3. Run all remaining cells from top to bottom.
4. The final cell starts the main experiment.
5. Generated files are written to `sigflow_outputs/`; downloaded prices are cached in `sigflow_cache/`.

The default experiment is designed to be practical. CRPS training regularisation, ablations and repeated walk-forward refits are implemented but disabled initially because they can require substantially more computation.

In [ ]:
# Environment setup
# Set this to False when the packages are already installed or when you want
# to install a specific CUDA-enabled PyTorch build yourself.
AUTO_INSTALL_MISSING_PACKAGES = True

import importlib.util
import subprocess
import sys

PACKAGE_IMPORTS = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "torch": "torch",
    "yfinance": "yfinance",
    "iisignature": "iisignature",
    "nflows": "nflows",
    "scikit-learn": "sklearn",
}

missing_packages = [
    package
    for package, import_name in PACKAGE_IMPORTS.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print("Missing packages:", ", ".join(missing_packages))
    if AUTO_INSTALL_MISSING_PACKAGES:
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", *missing_packages]
            )
            print("Installation complete.")
            print("If the next cell cannot import a newly installed package, restart the kernel once.")
        except subprocess.CalledProcessError as exc:
            raise RuntimeError(
                "Automatic installation failed. Install the listed packages manually, "
                "then rerun the notebook."
            ) from exc
    else:
        raise ModuleNotFoundError(
            "Install the missing packages, then rerun this cell: "
            + ", ".join(missing_packages)
        )
else:
    print("All required packages are available.")

In [ ]:
from __future__ import annotations

import copy
import json
import math
import random
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Iterable, Literal, Sequence

import iisignature
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import yfinance as yf
from nflows.distributions.normal import StandardNormal
from nflows.flows import Flow
from nflows.transforms.autoregressive import (
    MaskedPiecewiseRationalQuadraticAutoregressiveTransform,
)
from nflows.transforms.base import CompositeTransform
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
)
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=FutureWarning)

## Important configuration switches

The main settings live in the `Config` dataclass below.

For a first run, leave these disabled:

```python
use_crps_regulariser = False
run_ablation = False
run_walk_forward = False
```

After confirming the base model works, enable them selectively:

- `use_crps_regulariser=True` adds a sample-based proper-score term during training.
- `run_ablation=True` compares combined, signature-only and statistical-only features.
- `run_walk_forward=True` repeatedly rebuilds past-only splits and refits the model through the test period.

Other useful controls include the ticker list, dates, input window, forecast horizon, epochs, predictive sample count and walk-forward refit frequency.

## Configuration

In [ ]:
@dataclass(frozen=True)
class Config:
    # Data. yfinance's end date is exclusive.
    tickers: tuple[str, ...] = ("AAPL", "MSFT", "GOOGL", "AMZN")
    start_date: str = "2016-01-01"
    end_date: str = "2026-07-25"
    refresh_data: bool = False
    cache_dir: str = "sigflow_cache"
    output_dir: str = "sigflow_outputs"

    # Forecast construction.
    window: int = 20
    horizon: int = 10
    annualisation: float = 252.0
    ewma_lambda: float = 0.94

    # Signature path: time, cumulative standardised return, quadratic variation.
    signature_depth: int = 3
    use_logsignature: bool = True

    # Purged chronological split by forecast-origin date.
    train_fraction: float = 0.70
    validation_fraction: float = 0.15

    # Regime-conditioned spline-flow mixture.
    regimes: int = 3
    gate_hidden: int = 64
    gate_dropout: float = 0.10
    flow_layers: int = 2
    flow_hidden: int = 48
    flow_bins: int = 8
    flow_tail_bound: float = 5.0

    # Objective weights.
    expert_alignment_weight: float = 0.40
    regime_classification_weight: float = 0.30
    label_smoothing: float = 0.02

    # Optional proper-score regulariser. It is implemented but disabled by
    # default because sampling through every expert materially increases runtime.
    use_crps_regulariser: bool = False
    crps_max_weight: float = 0.05
    crps_warmup_fraction: float = 0.30
    crps_samples: int = 12

    # Optimisation.
    epochs: int = 150
    batch_size: int = 128
    learning_rate: float = 1e-4
    weight_decay: float = 1e-5
    gradient_clip: float = 1.0
    patience: int = 20
    min_delta: float = 1e-4
    scheduler_patience: int = 7
    scheduler_factor: float = 0.5
    num_workers: int = 0
    print_every: int = 10

    # Predictive simulation.
    prediction_samples: int = 512
    prediction_batch_size: int = 128

    # Optional expensive diagnostics.
    run_ablation: bool = False
    run_walk_forward: bool = False
    walk_forward_refit_every: int = 63  # roughly quarterly trading days
    walk_forward_train_years: int | None = 5
    walk_forward_epochs: int = 50
    walk_forward_validation_fraction: float = 0.15

    # Reproducibility.
    seed: int = 42


CFG = Config()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
REGIME_NAMES = np.array(["Low", "Medium", "High"])
EPS = 1e-8

print(f"Device: {DEVICE}")

## Reproducibility and utility classes

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


@dataclass
class Standardiser:
    mean: np.ndarray
    scale: np.ndarray

    @classmethod
    def fit(cls, x: np.ndarray) -> "Standardiser":
        mean = np.nanmean(x, axis=0)
        scale = np.nanstd(x, axis=0, ddof=0)
        scale = np.where(np.isfinite(scale) & (scale > 1e-6), scale, 1.0)
        return cls(mean=mean.astype(np.float32), scale=scale.astype(np.float32))

    def transform(self, x: np.ndarray) -> np.ndarray:
        z = (x - self.mean) / self.scale
        return np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    def to_dict(self) -> dict[str, list[float]]:
        return {"mean": self.mean.tolist(), "scale": self.scale.tolist()}


@dataclass
class MarketDataset:
    numeric_features: np.ndarray
    targets_log_vol: np.ndarray
    metadata: pd.DataFrame
    feature_names: list[str]


@dataclass
class PreparedSplit:
    train_indices: np.ndarray
    validation_indices: np.ndarray
    test_indices: np.ndarray
    train_cutoff: pd.Timestamp
    validation_cutoff: pd.Timestamp
    standardiser: Standardiser
    regime_thresholds: np.ndarray
    context_features: np.ndarray
    regime_labels: np.ndarray


set_seed(CFG.seed)

## Data download

In [ ]:
def _extract_close(data: pd.DataFrame, ticker: str) -> pd.Series:
    """Handle both single-level and MultiIndex yfinance outputs."""
    if data.empty:
        raise ValueError(f"No data returned for {ticker}.")

    if isinstance(data.columns, pd.MultiIndex):
        if "Close" not in data.columns.get_level_values(0):
            raise KeyError(f"Close column missing for {ticker}.")
        close = data["Close"]
        if isinstance(close, pd.DataFrame):
            if ticker in close.columns:
                close = close[ticker]
            else:
                close = close.iloc[:, 0]
    else:
        if "Close" not in data.columns:
            raise KeyError(f"Close column missing for {ticker}.")
        close = data["Close"]

    close = pd.Series(close, name="Close", dtype=float)
    close.index = pd.to_datetime(close.index).tz_localize(None)
    close = close[~close.index.duplicated(keep="last")].sort_index()
    close = close.replace([np.inf, -np.inf], np.nan).dropna()
    close = close[close > 0]
    return close


def load_close_series(ticker: str, cfg: Config) -> pd.Series:
    cache_dir = Path(cfg.cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_name = f"{ticker}_{cfg.start_date}_{cfg.end_date}.csv".replace("/", "-")
    cache_path = cache_dir / cache_name

    if cache_path.exists() and not cfg.refresh_data:
        cached = pd.read_csv(cache_path, index_col=0, parse_dates=True)
        if "Close" not in cached.columns:
            raise ValueError(f"Malformed cache file: {cache_path}")
        close = cached["Close"].astype(float).dropna()
        close.index = pd.to_datetime(close.index)
        return close

    data = yf.download(
        ticker,
        start=cfg.start_date,
        end=cfg.end_date,
        interval="1d",
        auto_adjust=True,
        repair=False,
        progress=False,
        threads=False,
        multi_level_index=False,
    )
    close = _extract_close(data, ticker)
    close.to_frame().to_csv(cache_path)
    return close

## Leakage-safe signature and statistical features

In [ ]:
class SignatureFeatureBuilder:
    def __init__(self, depth: int, use_logsignature: bool):
        self.depth = depth
        self.use_logsignature = use_logsignature
        self.path_dimension = 3
        self.prepared = iisignature.prepare(self.path_dimension, depth)

        dummy_path = np.zeros((3, self.path_dimension), dtype=np.float64)
        dummy_path[:, 0] = np.linspace(0.0, 1.0, 3)
        if use_logsignature:
            signature_size = len(iisignature.logsig(dummy_path, self.prepared))
            prefix = "logsig"
        else:
            signature_size = len(iisignature.sig(dummy_path, depth))
            prefix = "sig"

        self.signature_names = [f"{prefix}_{i}" for i in range(signature_size)]
        self.stat_names = [
            "stat_ann_mean_return",
            "stat_log_ann_vol",
            "stat_mean_abs_return",
            "stat_downside_ann_vol",
            "stat_upside_ann_vol",
            "stat_max_abs_return",
            "stat_last_return",
            "stat_cumulative_return",
            "stat_lag1_autocorrelation",
            "stat_skewness",
            "stat_excess_kurtosis",
            "stat_vol_of_vol",
            "stat_squared_return_trend",
        ]
        self.feature_names = self.signature_names + self.stat_names

    @staticmethod
    def _safe_autocorrelation(values: np.ndarray) -> float:
        if len(values) < 3:
            return 0.0
        left = values[:-1]
        right = values[1:]
        if np.std(left) < EPS or np.std(right) < EPS:
            return 0.0
        result = np.corrcoef(left, right)[0, 1]
        return float(result) if np.isfinite(result) else 0.0

    @staticmethod
    def _vol_of_vol(values: np.ndarray, subwindow: int = 5) -> float:
        if len(values) < subwindow:
            return 0.0
        rolling_mean_square = np.convolve(
            values**2,
            np.ones(subwindow, dtype=float) / subwindow,
            mode="valid",
        )
        return float(np.std(np.sqrt(np.maximum(rolling_mean_square, 0.0))))

    def _signature(self, past_returns: np.ndarray) -> np.ndarray:
        past_mean = float(np.mean(past_returns))
        past_scale = float(np.std(past_returns, ddof=1))
        past_scale = max(past_scale, EPS)
        standardised = (past_returns - past_mean) / past_scale

        # Prepend the origin so all paths begin at zero. The cumulative-return
        # channel is scaled by sqrt(window) and quadratic variation by window.
        time = np.linspace(0.0, 1.0, len(standardised) + 1)
        cumulative_return = np.concatenate(
            [[0.0], np.cumsum(standardised) / math.sqrt(len(standardised))]
        )
        quadratic_variation = np.concatenate(
            [[0.0], np.cumsum(standardised**2) / len(standardised)]
        )
        path = np.column_stack([time, cumulative_return, quadratic_variation])

        if self.use_logsignature:
            result = iisignature.logsig(path, self.prepared)
        else:
            result = iisignature.sig(path, self.depth)
        return np.asarray(result, dtype=np.float64)

    def _statistics(self, past_returns: np.ndarray, annualisation: float) -> np.ndarray:
        mean_return = float(np.mean(past_returns))
        sample_std = max(float(np.std(past_returns, ddof=1)), EPS)
        ann_vol = math.sqrt(annualisation) * sample_std

        downside = np.minimum(past_returns, 0.0)
        upside = np.maximum(past_returns, 0.0)
        downside_vol = math.sqrt(annualisation * float(np.mean(downside**2)))
        upside_vol = math.sqrt(annualisation * float(np.mean(upside**2)))

        z = (past_returns - mean_return) / sample_std
        skewness = float(np.mean(z**3))
        excess_kurtosis = float(np.mean(z**4) - 3.0)

        x = np.arange(len(past_returns), dtype=float)
        x_centered = x - x.mean()
        denominator = float(np.sum(x_centered**2))
        squared_return_trend = (
            float(np.sum(x_centered * (past_returns**2 - np.mean(past_returns**2))))
            / max(denominator, EPS)
        )

        return np.array(
            [
                annualisation * mean_return,
                math.log(ann_vol + EPS),
                float(np.mean(np.abs(past_returns))),
                downside_vol,
                upside_vol,
                float(np.max(np.abs(past_returns))),
                float(past_returns[-1]),
                float(np.sum(past_returns)),
                self._safe_autocorrelation(past_returns),
                skewness,
                excess_kurtosis,
                self._vol_of_vol(past_returns),
                squared_return_trend,
            ],
            dtype=np.float64,
        )

    def transform(self, past_returns: np.ndarray, annualisation: float) -> np.ndarray:
        signature = self._signature(past_returns)
        statistics = self._statistics(past_returns, annualisation)
        result = np.concatenate([signature, statistics])
        return np.nan_to_num(result, nan=0.0, posinf=0.0, neginf=0.0)


def annualised_realised_volatility(returns: np.ndarray, annualisation: float) -> float:
    return float(np.sqrt(annualisation * np.mean(np.asarray(returns, dtype=float) ** 2)))


def ewma_volatility(
    past_returns: np.ndarray,
    annualisation: float,
    decay: float,
) -> float:
    variance = float(np.var(past_returns, ddof=1))
    variance = max(variance, EPS)
    for value in past_returns:
        variance = decay * variance + (1.0 - decay) * float(value**2)
    return float(np.sqrt(annualisation * max(variance, EPS)))


def build_market_dataset(cfg: Config) -> MarketDataset:
    builder = SignatureFeatureBuilder(cfg.signature_depth, cfg.use_logsignature)
    feature_rows: list[np.ndarray] = []
    target_rows: list[float] = []
    records: list[dict[str, object]] = []

    for ticker_id, ticker in enumerate(cfg.tickers):
        close = load_close_series(ticker, cfg)
        log_returns = np.log(close).diff().dropna()
        values = log_returns.to_numpy(dtype=np.float64)
        dates = pd.DatetimeIndex(log_returns.index)

        minimum = cfg.window + cfg.horizon
        if len(values) < minimum:
            raise ValueError(
                f"{ticker} has only {len(values)} returns; at least {minimum} are required."
            )

        for i in range(cfg.window, len(values) - cfg.horizon + 1):
            past = values[i - cfg.window : i]
            future = values[i : i + cfg.horizon]

            actual_vol = annualised_realised_volatility(future, cfg.annualisation)
            rolling_vol = annualised_realised_volatility(past, cfg.annualisation)
            ewma_vol = ewma_volatility(past, cfg.annualisation, cfg.ewma_lambda)

            feature_rows.append(builder.transform(past, cfg.annualisation))
            target_rows.append(math.log(actual_vol + EPS))
            records.append(
                {
                    "ticker_id": ticker_id,
                    "ticker": ticker,
                    # Forecast is formed after observing the close at i - 1.
                    "origin_date": dates[i - 1],
                    "target_start_date": dates[i],
                    "target_end_date": dates[i + cfg.horizon - 1],
                    "actual_vol": actual_vol,
                    "rolling_vol_baseline": rolling_vol,
                    "ewma_vol_baseline": ewma_vol,
                }
            )

        print(f"{ticker}: {len(close):,} prices, {len(values):,} returns")

    metadata = pd.DataFrame.from_records(records)
    metadata = metadata.sort_values(["origin_date", "ticker"]).reset_index(drop=True)

    # Reorder arrays to match the sorted metadata. records were originally ticker-major.
    original_metadata = pd.DataFrame.from_records(records)
    original_metadata["_original_position"] = np.arange(len(original_metadata))
    ordering = (
        original_metadata.sort_values(["origin_date", "ticker"])["_original_position"]
        .to_numpy(dtype=int)
    )

    features = np.asarray(feature_rows, dtype=np.float32)[ordering]
    targets = np.asarray(target_rows, dtype=np.float32)[ordering]

    if not np.all(np.isfinite(features)) or not np.all(np.isfinite(targets)):
        raise ValueError("Non-finite values remain in the constructed dataset.")

    print(f"Total samples: {len(metadata):,}; numeric features: {features.shape[1]}")
    return MarketDataset(
        numeric_features=features,
        targets_log_vol=targets,
        metadata=metadata,
        feature_names=builder.feature_names,
    )

## Purged chronological split and training-only scaling

In [ ]:
def choose_cutoffs(metadata: pd.DataFrame, cfg: Config) -> tuple[pd.Timestamp, pd.Timestamp]:
    unique_dates = np.array(sorted(pd.to_datetime(metadata["origin_date"]).unique()))
    if len(unique_dates) < 20:
        raise ValueError("Not enough unique dates for a chronological split.")

    train_position = max(0, min(len(unique_dates) - 3, int(len(unique_dates) * cfg.train_fraction) - 1))
    validation_end_fraction = cfg.train_fraction + cfg.validation_fraction
    validation_position = max(
        train_position + 1,
        min(len(unique_dates) - 2, int(len(unique_dates) * validation_end_fraction) - 1),
    )
    return pd.Timestamp(unique_dates[train_position]), pd.Timestamp(unique_dates[validation_position])


def regime_labels_from_thresholds(
    targets_log_vol: np.ndarray,
    thresholds: np.ndarray,
) -> np.ndarray:
    return np.digitize(targets_log_vol, thresholds, right=False).astype(np.int64)


def build_context_features(
    numeric_features: np.ndarray,
    ticker_ids: np.ndarray,
    standardiser: Standardiser,
    number_of_tickers: int,
) -> np.ndarray:
    numeric = standardiser.transform(numeric_features)
    one_hot = np.eye(number_of_tickers, dtype=np.float32)[ticker_ids.astype(int)]
    return np.concatenate([numeric, one_hot], axis=1).astype(np.float32)


def prepare_split(dataset: MarketDataset, cfg: Config) -> PreparedSplit:
    metadata = dataset.metadata
    train_cutoff, validation_cutoff = choose_cutoffs(metadata, cfg)

    origin = pd.to_datetime(metadata["origin_date"])
    target_end = pd.to_datetime(metadata["target_end_date"])

    # Purging rule: every target in a split must be fully observable before the
    # next split begins. This removes overlapping future-return targets across boundaries.
    train_mask = target_end <= train_cutoff
    validation_mask = (origin > train_cutoff) & (target_end <= validation_cutoff)
    test_mask = origin > validation_cutoff

    train_indices = np.flatnonzero(train_mask.to_numpy())
    validation_indices = np.flatnonzero(validation_mask.to_numpy())
    test_indices = np.flatnonzero(test_mask.to_numpy())

    if min(len(train_indices), len(validation_indices), len(test_indices)) == 0:
        raise ValueError(
            "At least one split is empty. Adjust train_fraction, validation_fraction, "
            "window, horizon, or the date range."
        )

    standardiser = Standardiser.fit(dataset.numeric_features[train_indices])
    context = build_context_features(
        dataset.numeric_features,
        metadata["ticker_id"].to_numpy(),
        standardiser,
        len(cfg.tickers),
    )

    thresholds = np.quantile(dataset.targets_log_vol[train_indices], [1 / 3, 2 / 3])
    labels = regime_labels_from_thresholds(dataset.targets_log_vol, thresholds)

    metadata.loc[:, "split"] = "unused"
    metadata.loc[train_indices, "split"] = "train"
    metadata.loc[validation_indices, "split"] = "validation"
    metadata.loc[test_indices, "split"] = "test"
    metadata.loc[:, "regime"] = labels
    metadata.loc[:, "regime_name"] = REGIME_NAMES[labels]

    print(
        f"Train: {len(train_indices):,} through {train_cutoff.date()} | "
        f"Validation: {len(validation_indices):,} through {validation_cutoff.date()} | "
        f"Test: {len(test_indices):,}"
    )
    print(
        "Training regime boundaries (annualised volatility): "
        f"{math.exp(float(thresholds[0])):.3f}, {math.exp(float(thresholds[1])):.3f}"
    )

    return PreparedSplit(
        train_indices=train_indices,
        validation_indices=validation_indices,
        test_indices=test_indices,
        train_cutoff=train_cutoff,
        validation_cutoff=validation_cutoff,
        standardiser=standardiser,
        regime_thresholds=thresholds.astype(np.float32),
        context_features=context,
        regime_labels=labels,
    )

## Regime gate and nonlinear one-dimensional spline-flow experts

In [ ]:
class RegimeGate(nn.Module):
    def __init__(self, context_dimension: int, hidden: int, regimes: int, dropout: float):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(context_dimension, hidden),
            nn.LayerNorm(hidden),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, regimes),
        )

    def forward(self, context: torch.Tensor) -> torch.Tensor:
        # Return logits. log_softmax/CrossEntropyLoss are applied by the objective.
        return self.network(context)


def build_spline_flow(context_dimension: int, cfg: Config) -> Flow:
    transforms = []
    for _ in range(cfg.flow_layers):
        transforms.append(
            MaskedPiecewiseRationalQuadraticAutoregressiveTransform(
                features=1,
                hidden_features=cfg.flow_hidden,
                context_features=context_dimension,
                num_bins=cfg.flow_bins,
                tails="linear",
                tail_bound=cfg.flow_tail_bound,
                num_blocks=2,
                use_residual_blocks=True,
                dropout_probability=0.0,
                use_batch_norm=False,
            )
        )
    return Flow(CompositeTransform(transforms), StandardNormal([1]))


class RegimeSplineMixture(nn.Module):
    def __init__(self, context_dimension: int, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.gate = RegimeGate(
            context_dimension,
            cfg.gate_hidden,
            cfg.regimes,
            cfg.gate_dropout,
        )
        self.experts = nn.ModuleList(
            [build_spline_flow(context_dimension, cfg) for _ in range(cfg.regimes)]
        )

    def component_log_probabilities(
        self,
        context: torch.Tensor,
        target: torch.Tensor,
    ) -> torch.Tensor:
        return torch.stack(
            [expert.log_prob(target, context=context) for expert in self.experts],
            dim=1,
        )

    def mixture_log_probability(
        self,
        context: torch.Tensor,
        target: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        logits = self.gate(context)
        log_gate_probabilities = F.log_softmax(logits, dim=-1)
        component_log_probabilities = self.component_log_probabilities(context, target)
        mixture_log_probability = torch.logsumexp(
            log_gate_probabilities + component_log_probabilities,
            dim=1,
        )
        return mixture_log_probability, logits, component_log_probabilities

    def draw_component_samples(
        self,
        context: torch.Tensor,
        samples_per_expert: int,
    ) -> torch.Tensor:
        """Return log-volatility samples with shape [batch, regime, samples]."""
        draws = [
            expert.sample(samples_per_expert, context=context).squeeze(-1)
            for expert in self.experts
        ]
        return torch.stack(draws, dim=1)

    def draw_mixture_samples(
        self,
        context: torch.Tensor,
        number_of_samples: int,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Sample every trained expert according to the gate probabilities."""
        logits = self.gate(context)
        probabilities = F.softmax(logits, dim=-1)
        all_component_samples = self.draw_component_samples(context, number_of_samples)
        # Categorical.sample((S,)) -> [S, batch], then transpose to [batch, S].
        component_ids = torch.distributions.Categorical(probs=probabilities).sample(
            (number_of_samples,)
        ).transpose(0, 1)
        selected = torch.gather(
            all_component_samples.permute(0, 2, 1),
            dim=2,
            index=component_ids.unsqueeze(-1),
        ).squeeze(-1)
        return selected, probabilities


def crps_from_component_samples(
    component_samples: torch.Tensor,
    gate_probabilities: torch.Tensor,
    target: torch.Tensor,
) -> torch.Tensor:
    """Differentiable Monte-Carlo CRPS for a finite mixture.

    component_samples: [batch, components, samples]
    gate_probabilities: [batch, components]
    target: [batch, 1]
    """
    absolute_to_target = torch.abs(component_samples - target.unsqueeze(1))
    first_term = (
        gate_probabilities.unsqueeze(-1) * absolute_to_target
    ).sum(dim=1).mean(dim=1)

    left = component_samples[:, :, None, :, None]
    right = component_samples[:, None, :, None, :]
    pairwise_component_distance = torch.abs(left - right).mean(dim=(-1, -2))
    mixture_weights = gate_probabilities[:, :, None] * gate_probabilities[:, None, :]
    second_term = 0.5 * (mixture_weights * pairwise_component_distance).sum(dim=(1, 2))
    return (first_term - second_term).mean()


def objective_terms(
    model: RegimeSplineMixture,
    context: torch.Tensor,
    target: torch.Tensor,
    regime_label: torch.Tensor,
    cfg: Config,
    crps_weight: float = 0.0,
) -> dict[str, torch.Tensor]:
    mixture_log_probability, logits, component_log_probabilities = (
        model.mixture_log_probability(context, target)
    )
    mixture_nll = -mixture_log_probability.mean()
    selected_expert_nll = -component_log_probabilities.gather(
        1, regime_label.unsqueeze(1)
    ).mean()
    regime_cross_entropy = F.cross_entropy(
        logits,
        regime_label,
        label_smoothing=cfg.label_smoothing,
    )

    total = (
        mixture_nll
        + cfg.expert_alignment_weight * selected_expert_nll
        + cfg.regime_classification_weight * regime_cross_entropy
    )

    crps = torch.zeros((), device=context.device)
    if crps_weight > 0.0:
        component_samples = model.draw_component_samples(context, cfg.crps_samples)
        probabilities = F.softmax(logits, dim=-1)
        crps = crps_from_component_samples(component_samples, probabilities, target)
        total = total + crps_weight * crps

    regime_accuracy = (logits.argmax(dim=1) == regime_label).float().mean()
    return {
        "total": total,
        "mixture_nll": mixture_nll,
        "selected_expert_nll": selected_expert_nll,
        "regime_cross_entropy": regime_cross_entropy,
        "crps": crps,
        "regime_accuracy": regime_accuracy,
    }

## Training with validation, early stopping, checkpointing, and clipping

In [ ]:
def make_loader(
    context: np.ndarray,
    targets: np.ndarray,
    labels: np.ndarray,
    indices: np.ndarray,
    cfg: Config,
    shuffle: bool,
) -> DataLoader:
    dataset = TensorDataset(
        torch.from_numpy(context[indices]).float(),
        torch.from_numpy(targets[indices]).float().unsqueeze(1),
        torch.from_numpy(labels[indices]).long(),
    )
    return DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )


def crps_schedule(epoch: int, cfg: Config, total_epochs: int | None = None) -> float:
    if not cfg.use_crps_regulariser:
        return 0.0
    epochs = cfg.epochs if total_epochs is None else total_epochs
    warmup_epochs = max(1, int(epochs * cfg.crps_warmup_fraction))
    progress = min(1.0, max(0.0, epoch / warmup_epochs))
    return cfg.crps_max_weight * progress


def aggregate_batches(rows: list[dict[str, float]]) -> dict[str, float]:
    keys = rows[0].keys()
    return {key: float(np.mean([row[key] for row in rows])) for key in keys}


def run_epoch(
    model: RegimeSplineMixture,
    loader: DataLoader,
    cfg: Config,
    optimiser: optim.Optimizer | None,
    crps_weight: float,
) -> dict[str, float]:
    training = optimiser is not None
    model.train(training)
    rows: list[dict[str, float]] = []

    for context, target, label in loader:
        context = context.to(DEVICE, non_blocking=True)
        target = target.to(DEVICE, non_blocking=True)
        label = label.to(DEVICE, non_blocking=True)

        if training:
            optimiser.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            terms = objective_terms(
                model,
                context,
                target,
                label,
                cfg,
                crps_weight=crps_weight if training else 0.0,
            )
            if training:
                terms["total"].backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.gradient_clip)
                optimiser.step()

        rows.append({key: float(value.detach().cpu()) for key, value in terms.items()})

    return aggregate_batches(rows)


def fit_model(
    context: np.ndarray,
    targets: np.ndarray,
    labels: np.ndarray,
    train_indices: np.ndarray,
    validation_indices: np.ndarray,
    cfg: Config,
    epochs: int | None = None,
    patience: int | None = None,
) -> tuple[RegimeSplineMixture, dict[str, list[float]], int]:
    epochs = cfg.epochs if epochs is None else epochs
    patience = cfg.patience if patience is None else patience

    train_loader = make_loader(context, targets, labels, train_indices, cfg, shuffle=True)
    validation_loader = make_loader(
        context, targets, labels, validation_indices, cfg, shuffle=False
    )

    model = RegimeSplineMixture(context.shape[1], cfg).to(DEVICE)
    optimiser = optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimiser,
        mode="min",
        factor=cfg.scheduler_factor,
        patience=cfg.scheduler_patience,
    )

    history: dict[str, list[float]] = {
        "train_total": [],
        "train_mixture_nll": [],
        "validation_total": [],
        "validation_mixture_nll": [],
        "validation_regime_accuracy": [],
        "learning_rate": [],
    }

    best_value = float("inf")
    best_state: dict[str, torch.Tensor] | None = None
    best_epoch = 0
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):
        train_metrics = run_epoch(
            model,
            train_loader,
            cfg,
            optimiser,
            crps_weight=crps_schedule(epoch, cfg, epochs),
        )
        with torch.no_grad():
            validation_metrics = run_epoch(
                model,
                validation_loader,
                cfg,
                optimiser=None,
                crps_weight=0.0,
            )

        monitored_value = validation_metrics["total"]
        scheduler.step(monitored_value)

        history["train_total"].append(train_metrics["total"])
        history["train_mixture_nll"].append(train_metrics["mixture_nll"])
        history["validation_total"].append(validation_metrics["total"])
        history["validation_mixture_nll"].append(validation_metrics["mixture_nll"])
        history["validation_regime_accuracy"].append(
            validation_metrics["regime_accuracy"]
        )
        history["learning_rate"].append(float(optimiser.param_groups[0]["lr"]))

        improved = monitored_value < best_value - cfg.min_delta
        if improved:
            best_value = monitored_value
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epoch == 1 or epoch % cfg.print_every == 0 or epoch == epochs:
            print(
                f"Epoch {epoch:03d} | train {train_metrics['total']:.4f} | "
                f"val {validation_metrics['total']:.4f} | "
                f"val NLL {validation_metrics['mixture_nll']:.4f} | "
                f"regime acc {validation_metrics['regime_accuracy']:.3f}"
            )

        if epochs_without_improvement >= patience:
            print(f"Early stopping at epoch {epoch}; best epoch was {best_epoch}.")
            break

    if best_state is None:
        raise RuntimeError("Training did not produce a valid checkpoint.")
    model.load_state_dict(best_state)
    return model, history, best_epoch


def fit_fixed_epochs(
    context: np.ndarray,
    targets: np.ndarray,
    labels: np.ndarray,
    indices: np.ndarray,
    cfg: Config,
    epochs: int,
) -> RegimeSplineMixture:
    """Fit on all available past data after an epoch count was selected internally."""
    loader = make_loader(context, targets, labels, indices, cfg, shuffle=True)
    model = RegimeSplineMixture(context.shape[1], cfg).to(DEVICE)
    optimiser = optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )
    for epoch in range(1, epochs + 1):
        run_epoch(
            model,
            loader,
            cfg,
            optimiser,
            crps_weight=crps_schedule(epoch, cfg, epochs),
        )
    return model

## Proper mixture prediction, uncertainty, CRPS, PIT, and calibration

In [ ]:
def empirical_crps(samples: torch.Tensor, actual: torch.Tensor) -> torch.Tensor:
    """Memory-efficient empirical CRPS for samples shaped [batch, samples]."""
    number_of_samples = samples.shape[1]
    first_term = torch.mean(torch.abs(samples - actual.unsqueeze(1)), dim=1)
    sorted_samples, _ = torch.sort(samples, dim=1)
    ranks = torch.arange(
        1,
        number_of_samples + 1,
        device=samples.device,
        dtype=samples.dtype,
    )
    coefficients = 2.0 * ranks - number_of_samples - 1.0
    half_pairwise_term = (
        sorted_samples * coefficients.unsqueeze(0)
    ).sum(dim=1) / (number_of_samples**2)
    return first_term - half_pairwise_term


@torch.no_grad()
def predict_distribution(
    model: RegimeSplineMixture,
    context: np.ndarray,
    targets_log_vol: np.ndarray,
    metadata: pd.DataFrame,
    indices: np.ndarray,
    cfg: Config,
) -> pd.DataFrame:
    model.eval()
    output_rows: list[pd.DataFrame] = []

    for start in range(0, len(indices), cfg.prediction_batch_size):
        batch_indices = indices[start : start + cfg.prediction_batch_size]
        x = torch.from_numpy(context[batch_indices]).float().to(DEVICE)
        y = torch.from_numpy(targets_log_vol[batch_indices]).float().unsqueeze(1).to(DEVICE)

        log_vol_samples, probabilities = model.draw_mixture_samples(
            x, cfg.prediction_samples
        )
        volatility_samples = torch.exp(log_vol_samples)
        actual_volatility = torch.exp(y.squeeze(1))

        mixture_log_probability, logits, _ = model.mixture_log_probability(x, y)
        predicted_regime = logits.argmax(dim=1)

        quantiles = torch.quantile(
            volatility_samples,
            torch.tensor([0.05, 0.25, 0.50, 0.75, 0.95], device=DEVICE),
            dim=1,
        )
        crps = empirical_crps(volatility_samples, actual_volatility)
        pit = (volatility_samples <= actual_volatility.unsqueeze(1)).float().mean(dim=1)

        batch = metadata.iloc[batch_indices].copy().reset_index(drop=True)
        batch["predicted_mean_vol"] = volatility_samples.mean(dim=1).cpu().numpy()
        batch["predicted_std_vol"] = volatility_samples.std(dim=1).cpu().numpy()
        batch["predicted_q05_vol"] = quantiles[0].cpu().numpy()
        batch["predicted_q25_vol"] = quantiles[1].cpu().numpy()
        batch["predicted_median_vol"] = quantiles[2].cpu().numpy()
        batch["predicted_q75_vol"] = quantiles[3].cpu().numpy()
        batch["predicted_q95_vol"] = quantiles[4].cpu().numpy()
        batch["negative_log_likelihood"] = (-mixture_log_probability).cpu().numpy()
        batch["crps"] = crps.cpu().numpy()
        batch["pit"] = pit.cpu().numpy()
        batch["predicted_regime"] = predicted_regime.cpu().numpy()
        batch["predicted_regime_name"] = REGIME_NAMES[
            predicted_regime.cpu().numpy()
        ]
        for regime in range(cfg.regimes):
            batch[f"probability_{REGIME_NAMES[regime].lower()}"] = (
                probabilities[:, regime].cpu().numpy()
            )
        output_rows.append(batch)

    return pd.concat(output_rows, ignore_index=True)

## Metrics and transparent baselines

In [ ]:
def qlike_loss(actual_volatility: np.ndarray, predicted_volatility: np.ndarray) -> float:
    actual_variance = np.maximum(actual_volatility**2, EPS)
    predicted_variance = np.maximum(predicted_volatility**2, EPS)
    ratio = actual_variance / predicted_variance
    # This normalised form is zero at a perfect forecast.
    return float(np.mean(ratio - np.log(ratio) - 1.0))


def safe_correlation(left: np.ndarray, right: np.ndarray) -> float:
    if np.std(left) < EPS or np.std(right) < EPS:
        return float("nan")
    return float(np.corrcoef(left, right)[0, 1])


def point_metrics(actual: np.ndarray, predicted: np.ndarray) -> dict[str, float]:
    error = predicted - actual
    return {
        "mae": float(np.mean(np.abs(error))),
        "rmse": float(np.sqrt(np.mean(error**2))),
        "correlation": safe_correlation(actual, predicted),
        "qlike": qlike_loss(actual, predicted),
    }


def evaluate_predictions(predictions: pd.DataFrame, cfg: Config) -> tuple[pd.DataFrame, np.ndarray]:
    actual = predictions["actual_vol"].to_numpy(dtype=float)
    model_prediction = predictions["predicted_median_vol"].to_numpy(dtype=float)

    rows: list[dict[str, object]] = []
    model_row: dict[str, object] = {"model": "SigFlow spline mixture"}
    model_row.update(point_metrics(actual, model_prediction))
    model_row.update(
        {
            "negative_log_likelihood": float(
                predictions["negative_log_likelihood"].mean()
            ),
            "crps": float(predictions["crps"].mean()),
            "coverage_50": float(
                (
                    (actual >= predictions["predicted_q25_vol"].to_numpy())
                    & (actual <= predictions["predicted_q75_vol"].to_numpy())
                ).mean()
            ),
            "coverage_90": float(
                (
                    (actual >= predictions["predicted_q05_vol"].to_numpy())
                    & (actual <= predictions["predicted_q95_vol"].to_numpy())
                ).mean()
            ),
            "mean_50_interval_width": float(
                (
                    predictions["predicted_q75_vol"]
                    - predictions["predicted_q25_vol"]
                ).mean()
            ),
            "mean_90_interval_width": float(
                (
                    predictions["predicted_q95_vol"]
                    - predictions["predicted_q05_vol"]
                ).mean()
            ),
        }
    )

    true_regime = predictions["regime"].to_numpy(dtype=int)
    predicted_regime = predictions["predicted_regime"].to_numpy(dtype=int)
    model_row.update(
        {
            "regime_accuracy": float(accuracy_score(true_regime, predicted_regime)),
            "regime_balanced_accuracy": float(
                balanced_accuracy_score(true_regime, predicted_regime)
            ),
            "regime_macro_f1": float(
                f1_score(true_regime, predicted_regime, average="macro")
            ),
        }
    )

    probabilities = predictions[
        [f"probability_{name.lower()}" for name in REGIME_NAMES]
    ].to_numpy(dtype=float)
    one_hot = np.eye(cfg.regimes)[true_regime]
    model_row["regime_brier_score"] = float(
        np.mean(np.sum((probabilities - one_hot) ** 2, axis=1))
    )
    rows.append(model_row)

    for name, column in [
        ("Rolling-window volatility", "rolling_vol_baseline"),
        ("EWMA volatility", "ewma_vol_baseline"),
    ]:
        row: dict[str, object] = {"model": name}
        row.update(point_metrics(actual, predictions[column].to_numpy(dtype=float)))
        rows.append(row)

    matrix = confusion_matrix(true_regime, predicted_regime, labels=np.arange(cfg.regimes))
    return pd.DataFrame(rows).set_index("model"), matrix

## Visual diagnostics

In [ ]:
def plot_training_history(history: dict[str, list[float]]) -> None:
    epochs = np.arange(1, len(history["train_total"]) + 1)

    plt.figure(figsize=(11, 4))
    plt.plot(epochs, history["train_total"], label="Training objective")
    plt.plot(epochs, history["validation_total"], label="Validation objective")
    plt.plot(epochs, history["validation_mixture_nll"], label="Validation mixture NLL")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and validation history")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(11, 3.5))
    plt.plot(epochs, history["validation_regime_accuracy"])
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.ylim(0.0, 1.0)
    plt.title("Validation regime accuracy")
    plt.tight_layout()
    plt.show()


def plot_forecast_dashboard(
    predictions: pd.DataFrame, ticker: str, horizon: int
) -> None:
    data = predictions[predictions["ticker"] == ticker].sort_values("origin_date")
    if data.empty:
        raise ValueError(f"No predictions available for {ticker}.")

    dates = pd.to_datetime(data["origin_date"])
    plt.figure(figsize=(14, 5))
    plt.plot(dates, data["actual_vol"], label="Actual future volatility", linewidth=1.2)
    plt.plot(
        dates,
        data["predicted_median_vol"],
        label="Predicted median",
        linewidth=1.2,
    )
    plt.plot(
        dates,
        data["ewma_vol_baseline"],
        label="EWMA baseline",
        linewidth=0.9,
        alpha=0.8,
    )
    plt.fill_between(
        dates,
        data["predicted_q05_vol"],
        data["predicted_q95_vol"],
        alpha=0.20,
        label="90% predictive interval",
    )
    plt.ylabel("Annualised volatility")
    plt.title(f"{ticker}: {horizon}-day volatility forecast")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(14, 4))
    plt.stackplot(
        dates,
        data["probability_low"],
        data["probability_medium"],
        data["probability_high"],
        labels=REGIME_NAMES,
        alpha=0.8,
    )
    plt.ylim(0.0, 1.0)
    plt.ylabel("Probability")
    plt.title(f"{ticker}: forecast-regime probabilities")
    plt.legend(loc="upper left", ncol=3)
    plt.tight_layout()
    plt.show()


def plot_calibration(predictions: pd.DataFrame, matrix: np.ndarray) -> None:
    plt.figure(figsize=(8, 4))
    plt.hist(predictions["pit"], bins=np.linspace(0.0, 1.0, 11), edgecolor="black")
    plt.axhline(len(predictions) / 10.0, linestyle="--", linewidth=1.0)
    plt.xlabel("Probability integral transform")
    plt.ylabel("Count")
    plt.title("PIT calibration histogram")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(5, 4))
    plt.imshow(matrix, interpolation="nearest")
    plt.xticks(np.arange(len(REGIME_NAMES)), REGIME_NAMES)
    plt.yticks(np.arange(len(REGIME_NAMES)), REGIME_NAMES)
    plt.xlabel("Predicted regime")
    plt.ylabel("Actual regime")
    plt.title("Regime confusion matrix")
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            plt.text(column, row, str(matrix[row, column]), ha="center", va="center")
    plt.colorbar()
    plt.tight_layout()
    plt.show()


def plot_rolling_metrics(predictions: pd.DataFrame, ticker: str, width: int = 63) -> None:
    data = predictions[predictions["ticker"] == ticker].sort_values("origin_date").copy()
    data["absolute_error"] = np.abs(
        data["predicted_median_vol"] - data["actual_vol"]
    )
    data["rolling_mae"] = data["absolute_error"].rolling(width, min_periods=10).mean()
    data["rolling_crps"] = data["crps"].rolling(width, min_periods=10).mean()

    plt.figure(figsize=(14, 4))
    plt.plot(data["origin_date"], data["rolling_mae"], label="Rolling MAE")
    plt.plot(data["origin_date"], data["rolling_crps"], label="Rolling CRPS")
    plt.title(f"{ticker}: rolling {width}-observation forecast quality")
    plt.legend()
    plt.tight_layout()
    plt.show()

## Optional feature ablation

In [ ]:
def feature_indices_for_mode(
    feature_names: Sequence[str],
    mode: Literal["combined", "signature", "statistics"],
) -> np.ndarray:
    if mode == "combined":
        return np.arange(len(feature_names), dtype=int)
    if mode == "signature":
        return np.array(
            [i for i, name in enumerate(feature_names) if not name.startswith("stat_")],
            dtype=int,
        )
    if mode == "statistics":
        return np.array(
            [i for i, name in enumerate(feature_names) if name.startswith("stat_")],
            dtype=int,
        )
    raise ValueError(f"Unsupported feature mode: {mode}")


def run_ablation_study(dataset: MarketDataset, split: PreparedSplit, cfg: Config) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for mode in ("statistics", "signature", "combined"):
        print(f"\nAblation mode: {mode}")
        selected = feature_indices_for_mode(dataset.feature_names, mode)
        standardiser = Standardiser.fit(
            dataset.numeric_features[split.train_indices][:, selected]
        )
        context = build_context_features(
            dataset.numeric_features[:, selected],
            dataset.metadata["ticker_id"].to_numpy(),
            standardiser,
            len(cfg.tickers),
        )
        model, _, _ = fit_model(
            context,
            dataset.targets_log_vol,
            split.regime_labels,
            split.train_indices,
            split.validation_indices,
            cfg,
        )
        predictions = predict_distribution(
            model,
            context,
            dataset.targets_log_vol,
            dataset.metadata,
            split.test_indices,
            cfg,
        )
        metrics, _ = evaluate_predictions(predictions, cfg)
        model_metrics = metrics.loc["SigFlow spline mixture"].to_dict()
        model_metrics["feature_mode"] = mode
        rows.append(model_metrics)
    return pd.DataFrame(rows).set_index("feature_mode")

## Optional genuine walk-forward evaluation

In [ ]:
def internal_purged_split(
    metadata: pd.DataFrame,
    eligible_indices: np.ndarray,
    validation_fraction: float,
) -> tuple[np.ndarray, np.ndarray]:
    eligible = metadata.iloc[eligible_indices]
    dates = np.array(sorted(pd.to_datetime(eligible["origin_date"]).unique()))
    if len(dates) < 20:
        raise ValueError("Insufficient dates for an internal walk-forward validation split.")
    position = max(1, min(len(dates) - 2, int(len(dates) * (1.0 - validation_fraction))))
    cutoff = pd.Timestamp(dates[position])

    origin = pd.to_datetime(metadata["origin_date"])
    target_end = pd.to_datetime(metadata["target_end_date"])
    eligible_mask = np.zeros(len(metadata), dtype=bool)
    eligible_mask[eligible_indices] = True

    train = np.flatnonzero(eligible_mask & (target_end <= cutoff).to_numpy())
    validation = np.flatnonzero(eligible_mask & (origin > cutoff).to_numpy())
    if min(len(train), len(validation)) == 0:
        raise ValueError("Empty internal train or validation split.")
    return train, validation


def run_walk_forward_evaluation(
    dataset: MarketDataset,
    first_test_date: pd.Timestamp,
    cfg: Config,
) -> pd.DataFrame:
    """Quarterly-style refitting with no future target admitted before it is observable.

    For each block, an internal past-only validation tail selects the number of epochs.
    A fresh model is then trained for that epoch count on all eligible past samples and
    used to forecast the block. No future observation is used in scaling, regime
    thresholds, early stopping, or fitting.
    """
    metadata = dataset.metadata
    all_test_dates = np.array(
        sorted(
            pd.to_datetime(
                metadata.loc[metadata["origin_date"] >= first_test_date, "origin_date"]
            ).unique()
        )
    )
    outputs: list[pd.DataFrame] = []

    for start in range(0, len(all_test_dates), cfg.walk_forward_refit_every):
        block_dates = all_test_dates[start : start + cfg.walk_forward_refit_every]
        block_start = pd.Timestamp(block_dates[0])
        block_end = pd.Timestamp(block_dates[-1])

        eligible_mask = pd.to_datetime(metadata["target_end_date"]) < block_start
        if cfg.walk_forward_train_years is not None:
            earliest = block_start - pd.DateOffset(years=cfg.walk_forward_train_years)
            eligible_mask &= pd.to_datetime(metadata["origin_date"]) >= earliest
        eligible_indices = np.flatnonzero(eligible_mask.to_numpy())

        internal_train, internal_validation = internal_purged_split(
            metadata,
            eligible_indices,
            cfg.walk_forward_validation_fraction,
        )

        internal_standardiser = Standardiser.fit(
            dataset.numeric_features[internal_train]
        )
        internal_context = build_context_features(
            dataset.numeric_features,
            metadata["ticker_id"].to_numpy(),
            internal_standardiser,
            len(cfg.tickers),
        )
        internal_thresholds = np.quantile(
            dataset.targets_log_vol[internal_train], [1 / 3, 2 / 3]
        )
        internal_labels = regime_labels_from_thresholds(
            dataset.targets_log_vol, internal_thresholds
        )

        _, _, selected_epoch = fit_model(
            internal_context,
            dataset.targets_log_vol,
            internal_labels,
            internal_train,
            internal_validation,
            cfg,
            epochs=cfg.walk_forward_epochs,
            patience=max(8, cfg.patience // 2),
        )

        # Refit from scratch on every eligible past sample using training-only transforms.
        final_standardiser = Standardiser.fit(dataset.numeric_features[eligible_indices])
        final_context = build_context_features(
            dataset.numeric_features,
            metadata["ticker_id"].to_numpy(),
            final_standardiser,
            len(cfg.tickers),
        )
        final_thresholds = np.quantile(
            dataset.targets_log_vol[eligible_indices], [1 / 3, 2 / 3]
        )
        final_labels = regime_labels_from_thresholds(
            dataset.targets_log_vol, final_thresholds
        )
        final_model = fit_fixed_epochs(
            final_context,
            dataset.targets_log_vol,
            final_labels,
            eligible_indices,
            cfg,
            epochs=max(1, selected_epoch),
        )

        block_mask = (
            (pd.to_datetime(metadata["origin_date"]) >= block_start)
            & (pd.to_datetime(metadata["origin_date"]) <= block_end)
        )
        block_indices = np.flatnonzero(block_mask.to_numpy())

        block_metadata = metadata.copy()
        block_metadata.loc[:, "regime"] = final_labels
        block_metadata.loc[:, "regime_name"] = REGIME_NAMES[final_labels]
        block_predictions = predict_distribution(
            final_model,
            final_context,
            dataset.targets_log_vol,
            block_metadata,
            block_indices,
            cfg,
        )
        block_predictions["refit_date"] = block_start
        block_predictions["training_sample_count"] = len(eligible_indices)
        outputs.append(block_predictions)
        print(
            f"Walk-forward block {block_start.date()} to {block_end.date()}: "
            f"trained on {len(eligible_indices):,} samples for {selected_epoch} epochs"
        )

    return pd.concat(outputs, ignore_index=True)

## Save a reproducible checkpoint

In [ ]:
def save_outputs(
    model: RegimeSplineMixture,
    history: dict[str, list[float]],
    metrics: pd.DataFrame,
    predictions: pd.DataFrame,
    dataset: MarketDataset,
    split: PreparedSplit,
    cfg: Config,
) -> None:
    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    metrics.to_csv(output_dir / "test_metrics.csv")
    predictions.to_csv(output_dir / "test_predictions.csv", index=False)
    pd.DataFrame(history).to_csv(output_dir / "training_history.csv", index=False)

    checkpoint = {
        "model_state_dict": model.state_dict(),
        "config": asdict(cfg),
        "numeric_feature_names": dataset.feature_names,
        "ticker_order": list(cfg.tickers),
        "standardiser": split.standardiser.to_dict(),
        "regime_thresholds_log_vol": split.regime_thresholds.tolist(),
        "train_cutoff": str(split.train_cutoff.date()),
        "validation_cutoff": str(split.validation_cutoff.date()),
    }
    torch.save(checkpoint, output_dir / "sigflow_model.pt")
    with open(output_dir / "run_config.json", "w", encoding="utf-8") as handle:
        json.dump(asdict(cfg), handle, indent=2)

    print(f"Saved outputs to: {output_dir.resolve()}")

# Main experiment

In [ ]:
def main(cfg: Config = CFG) -> dict[str, object]:
    set_seed(cfg.seed)

    dataset = build_market_dataset(cfg)
    split = prepare_split(dataset, cfg)

    train_loader_context = split.context_features
    model, history, best_epoch = fit_model(
        train_loader_context,
        dataset.targets_log_vol,
        split.regime_labels,
        split.train_indices,
        split.validation_indices,
        cfg,
    )
    print(f"Best validation epoch: {best_epoch}")

    predictions = predict_distribution(
        model,
        train_loader_context,
        dataset.targets_log_vol,
        dataset.metadata,
        split.test_indices,
        cfg,
    )
    metrics, regime_matrix = evaluate_predictions(predictions, cfg)
    print("\nTest metrics")
    print(metrics.round(5).to_string())

    save_outputs(model, history, metrics, predictions, dataset, split, cfg)

    plot_training_history(history)
    for ticker in cfg.tickers:
        plot_forecast_dashboard(predictions, ticker, cfg.horizon)
        plot_rolling_metrics(predictions, ticker)
    plot_calibration(predictions, regime_matrix)

    ablation_metrics = None
    if cfg.run_ablation:
        ablation_metrics = run_ablation_study(dataset, split, cfg)
        print("\nAblation results")
        print(ablation_metrics.round(5).to_string())
        output_dir = Path(cfg.output_dir)
        ablation_metrics.to_csv(output_dir / "ablation_metrics.csv")

    walk_forward_predictions = None
    if cfg.run_walk_forward:
        first_test_date = pd.Timestamp(
            dataset.metadata.iloc[split.test_indices]["origin_date"].min()
        )
        walk_forward_predictions = run_walk_forward_evaluation(
            dataset,
            first_test_date,
            cfg,
        )
        walk_forward_metrics, _ = evaluate_predictions(walk_forward_predictions, cfg)
        print("\nWalk-forward metrics")
        print(walk_forward_metrics.round(5).to_string())
        output_dir = Path(cfg.output_dir)
        walk_forward_predictions.to_csv(
            output_dir / "walk_forward_predictions.csv", index=False
        )
        walk_forward_metrics.to_csv(output_dir / "walk_forward_metrics.csv")

    return {
        "dataset": dataset,
        "split": split,
        "model": model,
        "history": history,
        "predictions": predictions,
        "metrics": metrics,
        "regime_confusion_matrix": regime_matrix,
        "ablation_metrics": ablation_metrics,
        "walk_forward_predictions": walk_forward_predictions,
    }

Run this cell in a notebook. When executing the file directly, main() also runs.

# Run the experiment

Running the next cell will:

1. download or load cached adjusted closing prices;
2. construct ticker-safe samples and purged chronological splits;
3. fit the regime-conditioned probabilistic model;
4. evaluate it against transparent baselines;
5. display forecast, regime and calibration diagnostics;
6. save the model, predictions, metrics, training history and configuration.

To change settings, edit `CFG = Config(...)` in the configuration cell before running this cell.

In [ ]:
RUN_MAIN_EXPERIMENT = True

if RUN_MAIN_EXPERIMENT:
    RESULTS = main(CFG)
else:
    print("Set RUN_MAIN_EXPERIMENT = True when you are ready to train and evaluate.")

## Saved outputs and result objects

The notebook stores the returned experiment objects in `RESULTS`, including:

```python
RESULTS["dataset"]
RESULTS["split"]
RESULTS["model"]
RESULTS["history"]
RESULTS["predictions"]
RESULTS["metrics"]
RESULTS["regime_confusion_matrix"]
```

The default output directory contains:

- `sigflow_model.pt`
- `test_predictions.csv`
- `test_metrics.csv`
- `training_history.csv`
- `run_config.json`

Optional ablation and walk-forward CSV files are added when their switches are enabled.

### Reading the evaluation

The most useful checks are:

- whether the model improves on rolling-volatility and EWMA baselines;
- test negative log-likelihood and CRPS;
- 50%, 80% and 90% interval coverage;
- QLIKE, MAE and RMSE;
- PIT calibration;
- balanced regime accuracy and the confusion matrix;
- stability across tickers and through time.

A complex flow model should not be treated as useful unless it consistently improves on the simple baselines out of sample.